# Chapter 7.7: Feedback Loops & Long-term Optimization

## Learning Objectives

By the end of this notebook, you will be able to:

1. Understand how recommendation systems create echo chambers and filter bubbles
2. Model and simulate the popularity bias feedback loop (rich-get-richer)
3. Simulate feedback loop effects on user preferences over time
4. Implement interventions: diversity injection and exploration bonuses
5. Apply fairness constraints in the presence of feedback loops
6. Balance long-term user satisfaction with short-term engagement
7. Measure and visualize echo chamber formation

## Prerequisites

- Basic recommendation system concepts
- Understanding of collaborative filtering
- Familiarity with simulation methodology

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part7/chapter_7.7_feedback_loops.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/USERNAME/rec_system/raw/main/notebooks/part7/chapter_7.7_feedback_loops.ipynb)

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple
from collections import defaultdict
from copy import deepcopy

np.random.seed(42)
random.seed(42)

plt.style.use('seaborn-v0_8')
print("All imports successful.")

## 1. The Feedback Loop Problem

Recommendation systems create a **closed loop**:

```
User preferences -> Rec system -> Recommended items -> User interactions -> Updated preferences -> ...
```

This can lead to:

1. **Filter bubbles** (Pariser, 2011): Users only see content that confirms existing views
2. **Echo chambers**: User preferences narrow over time
3. **Popularity bias**: Popular items get more exposure, becoming more popular
4. **Homogenization**: All users converge to similar preferences

> **💡 Concept:** The feedback loop is a form of **distributional shift** where the training data distribution changes because of the model's own predictions. This is fundamentally different from the i.i.d. assumption in standard supervised learning.

### Mathematical Formulation

Let $p_t(i|u)$ be the probability of recommending item $i$ to user $u$ at time $t$.
User preferences evolve as:

$$\text{pref}_u^{t+1}(c) = (1-\alpha) \cdot \text{pref}_u^t(c) + \alpha \cdot \mathbb{I}[\text{consumed item of category } c]$$

If the recommender reinforces existing preferences, $\text{pref}_u^t$ concentrates on fewer categories over time.

In [ ]:
class FeedbackLoopSimulator:
    """Simulate feedback loops in recommendation systems."""
    
    def __init__(self, n_users: int = 100, n_items: int = 200, 
                 n_categories: int = 10, seed: int = 42):
        self.rng = np.random.RandomState(seed)
        self.n_users = n_users
        self.n_items = n_items
        self.n_categories = n_categories
        
        # Item properties
        self.item_categories = self.rng.randint(0, n_categories, n_items)
        self.item_quality = self.rng.beta(2, 2, n_items)
        self.item_popularity = np.zeros(n_items)  # Evolves over time
        
        # Initial popularity: power law
        self.item_popularity = self.rng.pareto(1.5, n_items)
        self.item_popularity /= self.item_popularity.sum()
        
        # User preferences: distribution over categories
        self.user_preferences = self.rng.dirichlet(np.ones(n_categories) * 2, n_users)
        self.initial_preferences = self.user_preferences.copy()
        
        # Interaction history
        self.interaction_counts = np.zeros((n_users, n_items))
        self.category_exposure = np.zeros((n_users, n_categories))
    
    def recommend(self, user_id: int, n_recs: int = 5, 
                  strategy: str = 'greedy') -> List[int]:
        """Generate recommendations using specified strategy."""
        scores = np.zeros(self.n_items)
        
        if strategy == 'greedy':
            # Pure exploitation: score = user_pref * item_quality
            for i in range(self.n_items):
                cat = self.item_categories[i]
                scores[i] = self.user_preferences[user_id, cat] * self.item_quality[i]
        
        elif strategy == 'popularity':
            # Popularity-biased
            for i in range(self.n_items):
                cat = self.item_categories[i]
                scores[i] = (self.user_preferences[user_id, cat] * 0.5 + 
                            self.item_popularity[i] * 0.5)
        
        elif strategy == 'diverse':
            # Diversity-aware: MMR-style
            for i in range(self.n_items):
                cat = self.item_categories[i]
                relevance = self.user_preferences[user_id, cat] * self.item_quality[i]
                # Penalize over-exposed categories
                exposure = self.category_exposure[user_id, cat]
                diversity_bonus = 1.0 / (1.0 + exposure * 0.1)
                scores[i] = relevance * 0.6 + diversity_bonus * 0.4
        
        elif strategy == 'epsilon_greedy':
            # Epsilon-greedy exploration
            if self.rng.random() < 0.2:
                return list(self.rng.choice(self.n_items, n_recs, replace=False))
            for i in range(self.n_items):
                cat = self.item_categories[i]
                scores[i] = self.user_preferences[user_id, cat] * self.item_quality[i]
        
        # Select top-K
        top_indices = np.argsort(scores)[-n_recs:][::-1]
        return list(top_indices)
    
    def simulate_interaction(self, user_id: int, item_id: int,
                              preference_update_rate: float = 0.05) -> bool:
        """Simulate user interaction and preference update."""
        cat = self.item_categories[item_id]
        pref = self.user_preferences[user_id, cat]
        quality = self.item_quality[item_id]
        
        # Click probability
        click_prob = pref * 0.6 + quality * 0.3 + 0.1
        click_prob = np.clip(click_prob, 0.05, 0.95)
        clicked = self.rng.random() < click_prob
        
        if clicked:
            # Update preferences: reinforce the clicked category
            self.user_preferences[user_id, cat] += preference_update_rate
            # Normalize to stay a distribution
            self.user_preferences[user_id] /= self.user_preferences[user_id].sum()
            
            # Update item popularity
            self.item_popularity[item_id] += 0.001
            self.item_popularity /= self.item_popularity.sum()
        
        # Track exposure
        self.category_exposure[user_id, cat] += 1
        self.interaction_counts[user_id, item_id] += 1
        
        return clicked
    
    def measure_echo_chamber(self) -> Dict:
        """Measure echo chamber metrics."""
        # Preference concentration (entropy)
        entropies = []
        for u in range(self.n_users):
            prefs = self.user_preferences[u]
            prefs = prefs[prefs > 0]  # Avoid log(0)
            entropy = -np.sum(prefs * np.log(prefs + 1e-10))
            entropies.append(entropy)
        
        # Preference shift from initial
        shifts = []
        for u in range(self.n_users):
            kl_div = np.sum(self.user_preferences[u] * 
                           np.log((self.user_preferences[u] + 1e-10) / 
                                  (self.initial_preferences[u] + 1e-10)))
            shifts.append(kl_div)
        
        # Gini coefficient of item popularity
        sorted_pop = np.sort(self.item_popularity)
        n = len(sorted_pop)
        cumulative = np.cumsum(sorted_pop)
        gini = (n + 1 - 2 * np.sum(cumulative) / cumulative[-1]) / n
        
        return {
            'avg_entropy': np.mean(entropies),
            'avg_kl_shift': np.mean(shifts),
            'popularity_gini': gini,
            'entropies': entropies
        }


sim = FeedbackLoopSimulator(n_users=100, n_items=200, seed=42)
initial_metrics = sim.measure_echo_chamber()
print(f"Initial state:")
print(f"  Avg preference entropy: {initial_metrics['avg_entropy']:.3f}")
print(f"  Popularity Gini: {initial_metrics['popularity_gini']:.3f}")

In [ ]:
def run_simulation(strategy: str, n_steps: int = 200, n_recs: int = 5, seed: int = 42):
    """Run feedback loop simulation with a given strategy."""
    sim = FeedbackLoopSimulator(n_users=100, n_items=200, seed=seed)
    metrics_over_time = []
    click_rates = []
    
    for step in range(n_steps):
        step_clicks = 0
        step_impressions = 0
        
        for user_id in range(sim.n_users):
            recs = sim.recommend(user_id, n_recs=n_recs, strategy=strategy)
            for item_id in recs:
                clicked = sim.simulate_interaction(user_id, item_id)
                step_clicks += int(clicked)
                step_impressions += 1
        
        click_rates.append(step_clicks / max(step_impressions, 1))
        
        if step % 20 == 0 or step == n_steps - 1:
            metrics = sim.measure_echo_chamber()
            metrics['step'] = step
            metrics['click_rate'] = click_rates[-1]
            metrics_over_time.append(metrics)
    
    return metrics_over_time, click_rates, sim


# Run for different strategies
strategies = ['greedy', 'popularity', 'diverse', 'epsilon_greedy']
all_results = {}

for strategy in strategies:
    print(f"\nSimulating: {strategy}")
    metrics, click_rates, sim = run_simulation(strategy, n_steps=200)
    all_results[strategy] = {
        'metrics': metrics,
        'click_rates': click_rates,
        'final_sim': sim
    }
    final = metrics[-1]
    print(f"  Final entropy: {final['avg_entropy']:.3f}, "
          f"KL shift: {final['avg_kl_shift']:.3f}, "
          f"Gini: {final['popularity_gini']:.3f}, "
          f"CTR: {final['click_rate']:.3f}")

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = {'greedy': 'steelblue', 'popularity': 'coral', 
          'diverse': 'green', 'epsilon_greedy': 'purple'}

# 1. Preference entropy over time
ax = axes[0, 0]
for strategy in strategies:
    steps = [m['step'] for m in all_results[strategy]['metrics']]
    entropies = [m['avg_entropy'] for m in all_results[strategy]['metrics']]
    ax.plot(steps, entropies, 'o-', label=strategy, color=colors[strategy])
ax.set_xlabel('Simulation Step')
ax.set_ylabel('Avg Preference Entropy')
ax.set_title('Echo Chamber Formation\n(Lower entropy = more concentrated preferences)')
ax.legend()

# 2. Popularity Gini over time
ax = axes[0, 1]
for strategy in strategies:
    steps = [m['step'] for m in all_results[strategy]['metrics']]
    ginis = [m['popularity_gini'] for m in all_results[strategy]['metrics']]
    ax.plot(steps, ginis, 'o-', label=strategy, color=colors[strategy])
ax.set_xlabel('Simulation Step')
ax.set_ylabel('Popularity Gini Coefficient')
ax.set_title('Popularity Concentration\n(Higher Gini = more inequality)')
ax.legend()

# 3. Click rates over time
ax = axes[1, 0]
window = 10
for strategy in strategies:
    cr = all_results[strategy]['click_rates']
    smoothed = np.convolve(cr, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, label=strategy, color=colors[strategy])
ax.set_xlabel('Simulation Step')
ax.set_ylabel('Click Rate')
ax.set_title('Click Rate Over Time')
ax.legend()

# 4. Final preference distributions (sample user)
ax = axes[1, 1]
user_id = 0
x = np.arange(all_results['greedy']['final_sim'].n_categories)
width = 0.2

for i, strategy in enumerate(strategies):
    prefs = all_results[strategy]['final_sim'].user_preferences[user_id]
    ax.bar(x + i * width, prefs, width, label=strategy, color=colors[strategy], alpha=0.7)

initial_prefs = all_results['greedy']['final_sim'].initial_preferences[user_id]
ax.step(x - 0.1, initial_prefs, where='mid', color='black', linewidth=2, 
        linestyle='--', label='Initial')
ax.set_xlabel('Category')
ax.set_ylabel('Preference Weight')
ax.set_title(f'User {user_id} Preference Distribution (Final)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 2. Interventions Against Filter Bubbles

Several interventions can mitigate feedback loop effects:

1. **Diversity injection**: Force minimum diversity in recommendations
2. **Exploration bonuses**: Add bonuses for unexplored items/categories
3. **Preference regularization**: Penalize extreme preference concentrations
4. **Exposure parity**: Ensure all items get minimum exposure

> **⚠️ Common Pitfall:** Too much intervention degrades relevance and user satisfaction. The key is finding the right balance between relevance and diversity.

In [ ]:
class InterventionSimulator(FeedbackLoopSimulator):
    """Simulator with diversity interventions."""
    
    def recommend_with_intervention(self, user_id: int, n_recs: int = 5,
                                     diversity_slots: int = 1,
                                     exploration_bonus: float = 0.0) -> List[int]:
        """Recommend with diversity intervention."""
        scores = np.zeros(self.n_items)
        
        for i in range(self.n_items):
            cat = self.item_categories[i]
            relevance = self.user_preferences[user_id, cat] * self.item_quality[i]
            
            # Exploration bonus for under-exposed categories
            total_exposure = self.category_exposure[user_id].sum() + 1
            cat_exposure = self.category_exposure[user_id, cat] + 1
            exp_bonus = exploration_bonus * np.log(total_exposure / cat_exposure)
            
            scores[i] = relevance + exp_bonus
        
        # Select main recommendations
        main_recs = list(np.argsort(scores)[-n_recs + diversity_slots:][::-1])
        
        # Add diversity slots: items from underexposed categories
        if diversity_slots > 0:
            exposure = self.category_exposure[user_id]
            underexposed = np.argsort(exposure)[:diversity_slots]
            
            for cat in underexposed:
                cat_items = np.where(self.item_categories == cat)[0]
                if len(cat_items) > 0:
                    cat_scores = self.item_quality[cat_items]
                    best = cat_items[np.argmax(cat_scores)]
                    if best not in main_recs:
                        main_recs.append(int(best))
        
        return main_recs[:n_recs]


def run_intervention_simulation(diversity_slots: int, exploration_bonus: float,
                                 n_steps: int = 200, seed: int = 42):
    """Run simulation with interventions."""
    sim = InterventionSimulator(n_users=100, n_items=200, seed=seed)
    metrics_over_time = []
    click_rates = []
    
    for step in range(n_steps):
        step_clicks = 0
        step_impressions = 0
        
        for user_id in range(sim.n_users):
            recs = sim.recommend_with_intervention(
                user_id, n_recs=5,
                diversity_slots=diversity_slots,
                exploration_bonus=exploration_bonus
            )
            for item_id in recs:
                clicked = sim.simulate_interaction(user_id, item_id)
                step_clicks += int(clicked)
                step_impressions += 1
        
        click_rates.append(step_clicks / max(step_impressions, 1))
        
        if step % 20 == 0 or step == n_steps - 1:
            metrics = sim.measure_echo_chamber()
            metrics['step'] = step
            metrics['click_rate'] = click_rates[-1]
            metrics_over_time.append(metrics)
    
    return metrics_over_time, click_rates


# Compare intervention levels
intervention_configs = [
    ('No intervention', 0, 0.0),
    ('1 diverse slot', 1, 0.0),
    ('2 diverse slots', 2, 0.0),
    ('Exploration bonus (0.3)', 0, 0.3),
    ('Combined', 1, 0.2),
]

intervention_results = {}
for name, d_slots, e_bonus in intervention_configs:
    metrics, crs = run_intervention_simulation(d_slots, e_bonus, n_steps=200)
    intervention_results[name] = {'metrics': metrics, 'click_rates': crs}
    final = metrics[-1]
    print(f"{name:30s} | Entropy: {final['avg_entropy']:.3f} | "
          f"Gini: {final['popularity_gini']:.3f} | CTR: {final['click_rate']:.3f}")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for name, data in intervention_results.items():
    steps = [m['step'] for m in data['metrics']]
    
    axes[0].plot(steps, [m['avg_entropy'] for m in data['metrics']], 'o-', label=name)
    axes[1].plot(steps, [m['popularity_gini'] for m in data['metrics']], 'o-', label=name)
    
    smoothed = np.convolve(data['click_rates'], np.ones(10)/10, mode='valid')
    axes[2].plot(smoothed, label=name)

axes[0].set_title('Preference Entropy (Higher = Less Echo Chamber)')
axes[0].set_xlabel('Step')
axes[0].legend(fontsize=7)

axes[1].set_title('Popularity Gini (Lower = More Fair)')
axes[1].set_xlabel('Step')
axes[1].legend(fontsize=7)

axes[2].set_title('Click Rate (User Engagement)')
axes[2].set_xlabel('Step')
axes[2].legend(fontsize=7)

plt.tight_layout()
plt.show()

## 3. Long-term vs Short-term Optimization

The fundamental tension:

- **Short-term**: Maximize immediate CTR/engagement -> leads to filter bubbles
- **Long-term**: Maximize user satisfaction over weeks/months -> requires exploration

$$J_{\text{total}} = \underbrace{(1-\lambda) \cdot J_{\text{short-term}}}_{\text{immediate CTR}} + \underbrace{\lambda \cdot J_{\text{long-term}}}_{\text{user retention}}$$

> **🔑 Pro Tip:** Netflix found that purely optimizing for play rate (short-term) led to recommending popular but average content. Adding long-term satisfaction signals like completion rate and return visits significantly improved overall user retention.

In [ ]:
class LongTermSimulator:
    """Simulate long-term user behavior with satisfaction and churn."""
    
    def __init__(self, n_users: int = 200, n_items: int = 100, seed: int = 42):
        self.rng = np.random.RandomState(seed)
        self.n_users = n_users
        self.n_items = n_items
        
        # User true preferences (hidden from recommender)
        self.true_prefs = self.rng.dirichlet(np.ones(10) * 2, n_users)
        self.item_cats = self.rng.randint(0, 10, n_items)
        self.item_quality = self.rng.beta(2, 2, n_items)
        
        # User state
        self.satisfaction = np.ones(n_users) * 0.5  # 0 to 1
        self.active = np.ones(n_users, dtype=bool)  # Whether user is still active
        self.novelty_need = np.ones(n_users) * 0.3  # Increases over time
        self.items_seen = [set() for _ in range(n_users)]
    
    def step(self, recommendations: Dict[int, List[int]], 
             strategy_name: str = '') -> Dict:
        """Simulate one time step."""
        clicks = 0
        impressions = 0
        churn_count = 0
        
        for user_id, rec_items in recommendations.items():
            if not self.active[user_id]:
                continue
            
            user_clicks = 0
            user_satisfaction_delta = 0
            
            for item_id in rec_items:
                impressions += 1
                cat = self.item_cats[item_id]
                pref = self.true_prefs[user_id, cat]
                quality = self.item_quality[item_id]
                
                # Novelty factor
                is_novel = item_id not in self.items_seen[user_id]
                novelty_bonus = 0.1 if is_novel else -0.05
                
                # Click probability
                click_prob = pref * 0.4 + quality * 0.3 + novelty_bonus + 0.1
                click_prob = np.clip(click_prob, 0.05, 0.95)
                
                if self.rng.random() < click_prob:
                    clicks += 1
                    user_clicks += 1
                    # Satisfaction depends on quality and novelty
                    user_satisfaction_delta += quality * 0.3 + novelty_bonus
                else:
                    user_satisfaction_delta -= 0.02  # Slight dissatisfaction for irrelevant recs
                
                self.items_seen[user_id].add(item_id)
            
            # Update satisfaction
            self.satisfaction[user_id] = np.clip(
                self.satisfaction[user_id] + user_satisfaction_delta * 0.1, 0, 1
            )
            
            # Novelty need increases over time
            self.novelty_need[user_id] = min(0.8, self.novelty_need[user_id] + 0.01)
            
            # Churn: users with low satisfaction may leave
            churn_prob = max(0, 0.3 - self.satisfaction[user_id] * 0.4)
            if self.rng.random() < churn_prob:
                self.active[user_id] = False
                churn_count += 1
        
        return {
            'click_rate': clicks / max(impressions, 1),
            'avg_satisfaction': np.mean(self.satisfaction[self.active]),
            'active_users': np.sum(self.active),
            'churn_count': churn_count,
            'retention_rate': np.mean(self.active)
        }


def run_longterm_comparison(n_steps: int = 100):
    """Compare short-term vs long-term optimization."""
    strategies = {
        'Short-term (greedy)': lambda sim, uid: sorted(
            range(sim.n_items),
            key=lambda i: sim.true_prefs[uid, sim.item_cats[i]] * sim.item_quality[i],
            reverse=True
        )[:5],
        'Long-term (diverse)': lambda sim, uid: _diverse_rec(sim, uid, 5),
        'Balanced': lambda sim, uid: _balanced_rec(sim, uid, 5),
    }
    
    results = {}
    for name, rec_fn in strategies.items():
        sim = LongTermSimulator(n_users=200, seed=42)
        history = []
        
        for step in range(n_steps):
            recs = {}
            for uid in range(sim.n_users):
                if sim.active[uid]:
                    recs[uid] = rec_fn(sim, uid)
            
            metrics = sim.step(recs)
            history.append(metrics)
        
        results[name] = history
        final = history[-1]
        print(f"{name:25s} | Retention: {final['retention_rate']:.1%} | "
              f"Satisfaction: {final['avg_satisfaction']:.3f} | CTR: {final['click_rate']:.3f}")
    
    return results


def _diverse_rec(sim, uid, k):
    """Diverse recommendation: include items from underexposed categories."""
    seen_cats = defaultdict(int)
    for item_id in sim.items_seen[uid]:
        seen_cats[sim.item_cats[item_id]] += 1
    
    scored = []
    for i in range(sim.n_items):
        cat = sim.item_cats[i]
        relevance = sim.true_prefs[uid, cat] * sim.item_quality[i]
        novelty = 1.0 if i not in sim.items_seen[uid] else 0.0
        diversity = 1.0 / (1.0 + seen_cats.get(cat, 0) * 0.2)
        score = relevance * 0.4 + novelty * 0.3 + diversity * 0.3
        scored.append((i, score))
    
    scored.sort(key=lambda x: -x[1])
    return [i for i, s in scored[:k]]


def _balanced_rec(sim, uid, k):
    """Balanced: mix of relevant and diverse."""
    # 3 relevant + 2 diverse
    relevant = sorted(
        range(sim.n_items),
        key=lambda i: sim.true_prefs[uid, sim.item_cats[i]] * sim.item_quality[i],
        reverse=True
    )[:3]
    
    diverse = _diverse_rec(sim, uid, k)
    diverse = [i for i in diverse if i not in relevant][:2]
    
    return relevant + diverse


lt_results = run_longterm_comparison(n_steps=100)

# Plot long-term comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for name, history in lt_results.items():
    axes[0].plot([h['retention_rate'] for h in history], label=name)
    axes[1].plot([h['avg_satisfaction'] for h in history], label=name)
    smoothed_ctr = np.convolve([h['click_rate'] for h in history], 
                                np.ones(5)/5, mode='valid')
    axes[2].plot(smoothed_ctr, label=name)

axes[0].set_title('User Retention Rate')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Retention')
axes[0].legend()

axes[1].set_title('Average User Satisfaction')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Satisfaction')
axes[1].legend()

axes[2].set_title('Click Rate (Smoothed)')
axes[2].set_xlabel('Step')
axes[2].set_ylabel('CTR')
axes[2].legend()

plt.tight_layout()
plt.show()

## 4. Summary

| Problem | Cause | Intervention | Trade-off |
|---------|-------|-------------|----------|
| Echo chambers | Preference reinforcement | Diversity injection | Relevance vs breadth |
| Popularity bias | Rich-get-richer | Exposure parity | Popular vs niche |
| User churn | Low satisfaction | Long-term optimization | Short-term vs retention |
| Homogenization | Common signals | Per-user exploration | Efficiency vs diversity |

**Key references:**
- Pariser (2011) - The Filter Bubble
- Jiang et al. (2019) - Degenerate Feedback Loops in Recommender Systems
- Mansoury et al. (2020) - Feedback Loop and Bias Amplification
- Chaney et al. (2018) - Algorithmic Effects on Diversity in Rec Systems

---

## Exercises

### 🏋️ Exercise 1: Simulate Echo Chamber and Measure Formation

In [ ]:
# 🏋️ Exercise 1: Detailed echo chamber analysis

def analyze_echo_chamber_formation(n_steps: int = 300):
    """Detailed analysis of echo chamber formation."""
    sim = FeedbackLoopSimulator(n_users=100, n_items=200, seed=42)
    
    # TODO: For each step:
    # 1. Run greedy recommendation
    # 2. Measure per-user preference entropy
    # 3. Track how many categories each user has been exposed to
    # 4. Compute user-user similarity (do users become more similar?)
    # 5. Identify "bubble" users (entropy < threshold)
    
    # TODO: Plot:
    # - Histogram of preference entropy at t=0, t=100, t=300
    # - Average categories exposed per user over time
    # - Fraction of "bubble" users over time
    
    pass

print("Exercise 1: Implement detailed echo chamber analysis")

### 🏋️ Exercise 2: Design an Adaptive Intervention

In [ ]:
# 🏋️ Exercise 2: Adaptive intervention that adjusts based on echo chamber severity

class AdaptiveInterventionPolicy:
    """Automatically adjusts diversity intervention based on echo chamber metrics."""
    
    def __init__(self, entropy_threshold: float = 1.5, 
                 max_diversity_slots: int = 3):
        self.entropy_threshold = entropy_threshold
        self.max_diversity_slots = max_diversity_slots
    
    def get_intervention_level(self, user_entropy: float) -> int:
        # TODO: Return number of diversity slots based on user's entropy
        # Low entropy (deep echo chamber) -> more intervention
        # High entropy (diverse) -> less intervention
        pass
    
    def recommend(self, sim, user_id: int, n_recs: int = 5) -> List[int]:
        # TODO: Compute user entropy and adapt recommendation accordingly
        pass

# TODO: Run simulation and show that adaptive policy balances CTR and diversity

print("Exercise 2: Implement AdaptiveInterventionPolicy")

### 🏋️ Exercise 3: Fairness-Aware Feedback Loop Correction

In [ ]:
# 🏋️ Exercise 3: Ensure fair exposure across item groups

class FairnessAwareRecommender:
    """Recommender that ensures fair exposure across categories."""
    
    def __init__(self, n_categories: int, fairness_constraint: float = 0.8):
        self.n_categories = n_categories
        self.fairness_constraint = fairness_constraint  # Min ratio of least to most exposed
        self.exposure_counts = np.zeros(n_categories)
    
    def recommend(self, user_scores: np.ndarray, item_categories: np.ndarray,
                  n_recs: int = 5) -> List[int]:
        # TODO: Select recommendations that:
        # 1. Maximize user relevance
        # 2. Subject to: min_category_exposure / max_category_exposure >= fairness_constraint
        # Use a greedy algorithm with fairness checking
        pass

# TODO: Run simulation comparing fair vs unfair recommender
# Plot exposure distribution over categories for both

print("Exercise 3: Implement FairnessAwareRecommender")